[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KalininGroup/camm_hackathon/blob/k4my4r/docs/day_4_11072025/CAMM_Hackathon_4.ipynb)


# CAMM Hackathon â€” DARA XRD Toolkit
2025-11-07

1. **Automated Refinement with DARA** (peak finding / whole-pattern assistance)
2. **Phase Search & Triage** (identify candidate phases prior to refinement)

**Jump to sections**  
- [Part 1 â€” Automated Refinement with DARA](#part-1)  
- [Part 2 â€” Phase Search with DARA](#part-2)  


***Data used in this notebook is from the DARA repository***

Link to Paper: https://arxiv.org/abs/2510.19667

Dara Repository: https://github.com/idocx/dara

This section walks you through **automated refinement using DARA** â€”  
an AI-driven assistant for analyzing X-ray diffraction (XRD) data.  
It helps identify key peaks, perform rapid matching, and suggest refinement parameters automatically.

Typical workflow:
1. Load or upload XRD data (e.g., `.xy` files)
2. Use DARA to find peak positions and intensities
3. Visualize and verify the refinement fit
4. Save results for phase search


In [1]:
# %% Setup (Colab)
# Uncomment and adapt as needed. Keep lightweight for hackathon use.
# !pip -q install numpy scipy matplotlib pandas scikit-learn plotly
# # Add domain-specific packages if required by your original notebooks:
# # !pip -q install pymatgen diffpy-cmi gsas2pkg

print('Environment check OK.')


Environment check OK.


# Part 1 â€” Automated Refinement with DARA
<a id='part-1'></a>


This section walks you through **automated refinement using DARA** â€”  
an AI-driven assistant for analyzing X-ray diffraction (XRD) data.  
It helps identify key peaks, perform rapid matching, and suggest refinement parameters automatically.

Typical workflow:
1. Load or upload XRD data (e.g., `.xy` files)
2. Use DARA to find peak positions and intensities
3. Visualize and verify the refinement fit
4. Save results for phase search


# Tutorial 1: Automated Refinement (with BGMN)
Dara provides a Python-based wrapper to the refinement software, BGMN, which implements a robust optimization algorithm that can refine automatically in most cases. This tutorial will show you how to interact with BGMN software and how to submit, adjust, and visualize your refinements.

> You can download this tutorial project from [here](https://idocx.github.io/dara/_static/tutorial.zip).

In [2]:
%pip install ipywidgets nbformat
!pip install dara-xrd
!pip install kaleido==0.2.1 # Install a compatible version of kaleido

Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 1.0.0 Requires-Python >=3.10; 1.1.0 Requires-Python >=3.10; 1.1.1 Requires-Python >=3.10; 1.1.2 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement dara-xrd (from versions: none)
ERROR: No matching distribution found for dara-xrd
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [3]:
!git clone https://github.com/idocx/dara.git # Sample data from DARA repository
!mv dara/notebooks/tutorial_data .
!rm -rf dara

fatal: Too many arguments.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --[no-]verbose    be more verbose
    -q, --[no-]quiet      be more quiet
    --[no-]progress       force progress reporting
    --[no-]reject-shallow don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --checkout            opposite of --no-checkout
    --[no-]bare           create a bare repository
    --[no-]mirror         create a mirror repository (implies --bare)
    -l, --[no-]local      to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    --hardlinks           opposite of --no-hardlinks
    -s, --[no-]shared     setup as shared repository
    --[no-]recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --[no-]recursive ...  alias of --recurse-submodules
    -j, --[no-]jobs <n>   number of submodules cloned in parallel
    --[no-]template <template-directory>
            

In [4]:
from pathlib import Path

from dara.refine import do_refinement_no_saving

ModuleNotFoundError: No module named 'dara'

In [ ]:
data = Path("tutorial_data")
cif_paths = list(data.glob("*.cif"))  # include all the cif files in the data folder

pattern_fn = "CaNi(PO3)4_800_240_Ca(OH)2_(NH4)2HPO4_NiO.xy"

## Basic Refinement
A one-line refinement with the default settings.

### Do refinement
By running `do_refinement_no_saving`, the refinement will be performed and the results
will be printed out. There will be no BGMN refinement output folder saved on the disk.

The only two things you will need to feed into the system:
- the path to the pattern. Currently, Dara only supports the `xy`, `xrdml`, `raw` formats.
- a list of CIF file paths. The CIF will be used as the reference structure for the refinement.

In [ ]:
refinement = do_refinement_no_saving(data / pattern_fn, cif_paths)

### Visualization
You can call `visualize` to visualize the refinement results. The observed, calculated, and difference patterns will be plotted.

In [ ]:
refinement.visualize()

#### Save the refinement plot
Optionally, if you want to share the plot with others, you can save the plot by calling `write_image` or `write_html` in the `plotly.Figure` object returned by `.visualize()`. The plot will be saved on the disk.

In [ ]:
refinement.visualize().write_html("tutorial_refinement.html")  # output the interactive html file to the disk
refinement.visualize().write_image("tutorial_refinement.png")  # output the png image to the disk

### Extracting information from the refinement

After finishing refinement, you can read information from the `RefinementResult` object. The object contains the following attributes:
- `lst_data`: information about phases, metrics of refinement (from the .lst file in BGMN)
- `peak_data`: the simulated peaks in the calculated pattern
- `plot_data`: `x` (two-theta), `y_obs`, `y_calc`, `y_bkg`, contribution from each phase. This is mainly used for visualization.

For example, you can get Rwp from `lst_data`

In [ ]:
f"The refinement has Rwp = {refinement.lst_data.rwp} %"

You can also get the information about the lattice, weight fraction for the phase.

Some values are in a tuple, with first value as the value and the second value as the error.

In [ ]:
phase_name = "CaNi(PO3)4_15_sym"
phase_result = refinement.lst_data.phases_results[phase_name]

gewicht = phase_result.gewicht
lattice_a = phase_result.a
lattice_b = phase_result.b
lattice_c = phase_result.c
lattice_alpha = phase_result.alpha
lattice_beta = phase_result.beta
lattice_gamma = phase_result.gamma

print(f"The lattice parameters of the phase {phase_name} are:\n" \
f"    a = {lattice_a} nm, b = {lattice_b} nm, c = {lattice_c} nm,\n"  \
f"    alpha = {lattice_alpha}, beta = {lattice_beta}, gamma = {lattice_gamma}\n")
print(f"The weight fraction of the phase {phase_name} is ({gewicht[0]} Â± {gewicht[1]}) %")

Peak data is stored in a pandas `DataFrame`.

In [ ]:
refinement.peak_data

#### Export the refined structure
You can also export the refined structure as a `pymatgen.Structure` object.

In [ ]:
structure = refinement.export_structure("CaNi(PO3)4_15_sym")
print(structure.to("tutorial_refinement_refined_CaNi(PO3)4_15_sym.cif", symprec=1e-3))

## Refining with customized phase parameters
The refinement with default setting looks good. But can it be better?

Dara supports the basic refinement parameters in BGMN / Profex. You can adjust the refinement parameters by passing the parameters to the `do_refinement_no_saving` function.

Common parameters include:
- `lattice_range`: you can (and need to) specify the range that the lattice parameters can vary. Usually, it can be a small range, like 0.01 ~ 0.05. It is applied to all lattice parameters (a, b, c, alpha, beta, gamma).
- `b1`: controls the width of the peak. `b1` describes the average particle size in the
  XRD sample. The larger the `b1`, the broader the peak. Usually, it is constraint to a
  small range, like from 0 to 0.005. If the fitted  `b1` is too large, you will see the peaks
  go too broad. In this case, your simulated pattern will look like an amorphous
  material that can be easily fit into the background.
- `k1`: also controls the width of the peak. `k1` describes the width of the particle size distribution in the sample. The larger the `k1`, the smaller the distribution is. Usually, it can be constrained to 0 ~ 1.
- `k2`: describes the microstrain in the sample. The larger the `k2`, the larger the microstrain. Usually, it can be a fixed value, like 0.
- `gewicht`: means "weight" in German. it contains the information of scale factor. However, in BGMN, it can also be used to specify the preferred orientation you would like to use in the refinement. By specifying the preferred orientation, you can vary the intensity of a set of reflections in the pattern, which can help you fit your pattern better. BGMN is able to decide which reflection to adjust automatically. You only need to specify how strong the preferred orientation is. Usually, it can be `SPHAR0` (none), `SPHAR2` (two preferred orientation parameters), or `SPHAR4` (four preferred orientation parameters), ... (up to `SPHAR10`). The larger the `gewicht`, the stronger the preferred orientation is. But it can cause overfitting as well.

#### Input parameter format
In Dara, all the phase parameters are passed as a dictionary arg called `phase_params`. The key is the parameter name, and the value is the parameter value. Dara supports three types of values:

- `fixed`. This is a string. The parameter will be fixed to the default value (usually 0).
- `(initial value)_(min value)^(max value)`. This is a string. The parameter will be
  allowed to vary in the refinement between initial value to the min value to the max value. The min value begins with
  `_`, and the max value begins with `^`.
- Other values. It can be a string or a number. For example, setting `gewicht` to
  `SPHAR2` means that preferred orientation is modeled with the `SPHAR2` settings; setting `lattice_range` to
  0.05 means that the lattice parameters can vary up to 5%.. See the BGMN / Profex
  manual for more information on these settings.

If you want to allow 5% variation in lattice parameters, then use the following
settings. `b1`: (started from 0, min = 0, max = 0.005), `k1`: (started from 0, min = 0,
max = 1), `k2`: fixed to 0, and `gewicht`: `SPHAR2`. This corresponds to a
`phase_params` dict of:

In [ ]:
phase_params = {
    "lattice_range": 0.05,
    "b1": "0_0^0.005",
    "k1": "0_0^1",
    "k2": "fixed",
    "gewicht": "SPHAR2"
}

In [ ]:
refinement = do_refinement_no_saving(data / pattern_fn, cif_paths, phase_params=phase_params)

In [ ]:
refinement.visualize()

Now you can see the Rwp of the refinement is slightly lower, indicating a better fit.

#### Specify different parameters for different phases
In the previous example, the refinement option is applied to all phases. But you can also specify different parameters for different phases. To do so, you will need to pass a special `RefinementPhase` object to the `phases` parameter.

In [ ]:
from dara import RefinementPhase

phases = [RefinementPhase.make(cif_path) for cif_path in cif_paths]

for phase in phases:
    # use a smaller lattice range for each phase
    phase.params["lattice_range"] = 0.01


refinement = do_refinement_no_saving(
    data / pattern_fn,
    phases=phases,
    # if one parameter is both specified in phase_params and in the RefinementPhase object, the value in RefinementPhase will be used.
    phase_params={
        "lattice_range": 0.05,  # <- this will be ignored because it has already been set in the eahc RefinementPhase object
        "b1": "0_0^0.005",
        "k1": "0_0^1",
        "k2": "fixed",
        "gewicht": "SPHAR2"
    }
)

## Refining with different instrument profiles, angle range, wavelength, etc.

If you would like to do refinement in a different instrument profiles or angle range, you can specify it in the refinement function as well.

- `instrument_name`: the instrument profile you would like to use. The default instrument
  profile used is *Aeris-fds-Pixcel1d-Medipix3*. You can find the available instrument profiles in the
  `dara/data/BGMN-Templates/Devices` folder.
- `wavelength`: the wavelength you would like to use in the refinement. It can be two types:
  - a number: the wavelength in nm. It is useful when you analyzing the data from a synchrotron.
  - a string: the element symbol. It represents the target material in the X-ray tube. BGMN can automatically find the distribution of the wavelength for the given metal. Currently, it supports sources of ["Cu", "Co", "Cr", "Fe", "Mo"]
- `wmin`, `wmax`: the angle range you would like to use in the refinement. It is set in `refinement_params`.

In [ ]:
refinement = do_refinement_no_saving(
    data / pattern_fn,
    cif_paths,
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
    wavelength="Cu",
    refinement_params={
        "wmin": 20,  # set the minimum two-thera in the refinement to be 20 deg.
        "wmax": 50  # set the maximum two-theta in the refinement to be 50 deg.
    }
)

In [ ]:
refinement.visualize()

## Save the project file to a folder on the disk
Usually, you don't have to directly interact with the BGMN input/output files.

However, if you would like to save the project directory for later use, you can use the
`do_refinement` function instead.

The refinement files will be saved in the path specified by `working_dir`. Other than
that, `do_refinement` and `do_refinement_no_saving` share the same parameters and
output.

Feel free to modify the refinement project file yourself or by loading with the Profex software.

In [ ]:
from dara import do_refinement

refinement = do_refinement(data / pattern_fn, cif_paths, working_dir="tutorial_refinement")

In [ ]:
refinement_folder = Path("tutorial_refinement")

# show all the files in the folder
for file in refinement_folder.glob("*"):
    print(">", file.name)

---

# Part 2 â€” Phase Search with DARA
<a id='part-2'></a>


This section focuses on **phase search and triage** â€” identifying possible crystal structures  
that match experimental data before detailed refinement.

Typical workflow:
1. Provide a chemical formula or database path
2. Let DARA (or external sources like Materials Project) fetch possible phases
3. Compare simulated vs. measured diffraction patterns
4. Narrow down candidate phases for refinement


# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In this tutorial, we will try to identify the phases in one experimental solid-state
reaction sample between `GeO2` and `ZnO`.

> You can download this tutorial project from [here](https://idocx.github.io/dara/_static/tutorial.zip).

In [ ]:
%pip install ipywidgets nbformat
!pip install dara-xrd
!pip install kaleido==0.2.1 # Install a compatible version of kaleido


In [ ]:
from pathlib import Path

from dara import search_phases

In [ ]:
!git clone https://github.com/idocx/dara.git
!mv dara/notebooks/tutorial_data .
!rm -rf dara

In [ ]:
pattern_path = "tutorial_data/GeO2-ZnO_700C_60min.xrdml"

# three elements are present in the sample
chemical_system = "Ge-O-Zn"

## Step 1: Prepare reference phases

Dara pre-builds an index of all the unique and low-energy phases in ICSD and COD
databases. It also implements a method to download CIF structures from COD data server
so that there is no need to obtain the offline database.

Before every search, we will need to gather all the reference phases in the chemical
system for the search algorithm. Dara provides `ICSDDatabase` and `CODDatabase` to do
the filtering.

In this example, we will use `CODDatabase` to download all the phases in the chemical system of `Ge-O-Zn`.

In [ ]:
from dara.structure_db import CODDatabase

# The COD database contains methods to filter phases in the chemical system
cod_database = CODDatabase()

# gather reference phases and save them to a directory called "cifs"
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In this case, we are using the XRD pattern from the solid-state reaction sample
on our laboratory's Aeris diffractometer (`tutorial_data/GeO2-ZnO_700C_60min.xrdml`).

In [ ]:
# gather all the phases in the "cifs" directory
all_cifs = list(Path("cifs").glob("*.cif"))

search_results = search_phases(
    pattern_path=pattern_path,
    phases=all_cifs,
    wavelength="Cu",
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
)

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [ ]:
search_results

In this pattern, we only have one solution found with `Rwp = 12.11 %`.

In [ ]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [ ]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [ ]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")

From the result, you can see that for the phase `GeO2`, the algorithm identifies two
similar phases with slightly different spacegroups (152 and 154).

---